# 04 · Evaluate all models
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mundmina/dl_project/blob/main/notebooks/04_evaluate.ipynb)

Оценивает каждый `runs/*_seed*/best.pt` и `runs/hog_svm/model.joblib` в условиях
original / swapped / fgonly / bgonly и собирает главную таблицу (mean ± std по seed).
Уже посчитанные предсказания берутся из `MyDrive/bgbias/results/<split>/`, новые модели (ablation) досчитываются.
В конце результаты автоматически пушатся в GitHub.

**`SPLIT = "test"`** — финальная оценка. Валидационный прогон сделан 2026-10-22 (`results/val/`), конфиги
зафиксированы, все seed обучены. После test ничего не подбираем и не переобучаем.

In [ ]:
SPLIT = "test"   # "val" for development runs

In [ ]:
# Shared setup: Drive, repo, dependencies. Same cell in every notebook.
# Colab: nothing to set. Local machine (own GPU): see LOCAL.md — open this notebook from the repo's notebooks/
# folder and set BGBIAS_DRIVE, BGBIAS_DATA and GITHUB_TOKEN before starting Jupyter.
REPO_URL = "https://github.com/mundmina/dl_project.git"
BRANCH = "main"

import os, subprocess, sys
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/dl_project"
    drive.mount("/content/drive")
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:  # always run the latest code from GitHub
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebook opened from <repo>/notebooks
    assert os.path.exists(os.path.join(REPO_DIR, "src", "bgbias")), "open the notebook from the repo's notebooks/ folder"
    missing = [v for v in ("BGBIAS_DRIVE", "BGBIAS_DATA") if not os.environ.get(v)]
    assert not missing, f"set {missing} before starting Jupyter (see LOCAL.md)"
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("code version:", commit)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
# DataLoader workers on Windows start fresh interpreters: they find bgbias through PYTHONPATH, not sys.path
os.environ["PYTHONPATH"] = os.pathsep.join(p for p in (f"{REPO_DIR}/src", os.environ.get("PYTHONPATH")) if p)
try:  # ask for secret access now, not at the end of a long run (the prompt times out unattended)
    if not os.environ.get("GITHUB_TOKEN"):
        from google.colab import userdata
        userdata.get("GITHUB_TOKEN")
    print("GITHUB_TOKEN: ok")
except Exception as e:  # noqa: BLE001
    print(f"GITHUB_TOKEN not available ({type(e).__name__}): results will not be pushed to GitHub")

In [ ]:
from pathlib import Path
from bgbias.project import load_project_data
from bgbias.utils import DRIVE_ROOT

data = load_project_data(REPO_DIR)   # splits / manifests from the repo or from MyDrive/bgbias/week1
RUNS = DRIVE_ROOT / "runs"           # checkpoints and per-run outputs live on Drive
RUNS.mkdir(parents=True, exist_ok=True)
{name: len(df) for name, df in data.splits.items()}

In [ ]:
import joblib
import pandas as pd
import torch
from bgbias import evaluate as E, hog_svm
from bgbias.models import load_checkpoint
from bgbias.project import make_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path(REPO_DIR) / "results" / SPLIT
OUT.mkdir(parents=True, exist_ok=True)
summaries = {}

CACHE = DRIVE_ROOT / "results" / SPLIT   # predictions already computed in an earlier run are reused

for run in sorted(p.parent for p in RUNS.glob("*_seed*/best.pt")):
    cached = CACHE / f"{run.name}_predictions.csv"
    if cached.exists():
        table = pd.read_csv(cached)
    else:
        model, cfg = load_checkpoint(run / "best.pt", data.num_classes, device)
        predict = E.torch_predict_fn(model, data, SPLIT, device, include_border=cfg["include_border"],
                                     blur_sigma=cfg["blur_sigma"], img_size=cfg["img_size"])
        table = E.predictions_table(data, SPLIT, predict)
    table.to_csv(OUT / f"{run.name}_predictions.csv", index=False)
    summaries[run.name] = E.summarize(table)
    print(run.name, {k: round(v, 4) for k, v in summaries[run.name].items() if k.startswith(("acc", "gap"))})

hog_path = RUNS / "hog_svm" / "model.joblib"
if (CACHE / "hog_svm_predictions.csv").exists():
    table = pd.read_csv(CACHE / "hog_svm_predictions.csv")
elif hog_path.exists():
    clf = joblib.load(hog_path)
    predict = lambda c: clf.predict(hog_svm.extract(make_dataset(data, SPLIT, condition=c, transform=None)))
    table = E.predictions_table(data, SPLIT, predict)
if hog_path.exists():
    table.to_csv(OUT / "hog_svm_predictions.csv", index=False)
    summaries["hog_svm"] = E.summarize(table)
    print("hog_svm", {k: round(v, 4) for k, v in summaries["hog_svm"].items() if k.startswith(("acc", "gap"))})

## Главная таблица

In [ ]:
per_run = pd.DataFrame(summaries).T
per_run.to_csv(OUT / "summary_per_run.csv")

COLS = ["acc_original", "acc_swapped", "acc_fgonly", "acc_bgonly", "f1_original", "f1_swapped",
        "gap_swap", "gap_fg", "flip_swap"]
model = per_run.index.str.replace(r"_seed\d+$", "", regex=True)
mean = per_run.groupby(model)[COLS].mean()
std = per_run.groupby(model)[COLS].std().fillna(0)
main = mean.map(lambda v: f"{v:.3f}") + " ± " + std.map(lambda v: f"{v:.3f}")
main["seeds"] = per_run.groupby(model).size()
main.to_csv(OUT / "main_table.csv")
main

In [ ]:
import shutil
from bgbias.github_sync import sync

dst = DRIVE_ROOT / "results" / SPLIT
shutil.copytree(OUT, dst, dirs_exist_ok=True)
print("saved to", OUT, "and", dst)
sync(REPO_DIR, f"Evaluation results ({SPLIT})")